### Dataset Explorations 

##### Step 1: Schema, Dtypes & Missingness Check

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("../data/ielts_task2_train.parquet")
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
print(df.dtypes)
print("\nMissing values:\n", df.isnull().sum())

Dataset Shape: 8049 rows, 11 columns

essay_id                    object
topic                       object
question                    object
essay_text                  object
gpt_advices                 object
gpt_positive_highlights     object
overall_band               float64
task_achievement_band      float64
coherence_cohesion_band    float64
lexical_resource_band        int64
grammatical_range_band       int64
dtype: object

Missing values:
 essay_id                   0
topic                      0
question                   0
essay_text                 0
gpt_advices                0
gpt_positive_highlights    0
overall_band               0
task_achievement_band      0
coherence_cohesion_band    0
lexical_resource_band      0
grammatical_range_band     0
dtype: int64


##### Step 2: Payload Inspection (What is actually inside each column?)

In [3]:
sample = df.iloc[0]
for col in df.columns:
    val = sample[col]
    val_repr = repr(val[:100]) if isinstance(val, (str, list, np.ndarray)) else val
    print(f"{col:25} -> {val_repr}")

essay_id                  -> '66abd675d091c90011b44e19'
topic                     -> 'Art'
question                  -> 'In many countries, people now wear western clothes such as suits and jeans rather than traditional c'
essay_text                -> 'In some nations, individuals tend to use Western-style attire instead of traditional dress. In my op'
gpt_advices               -> array(['To improve coherence and cohesion, ensure that your ideas flow logically and are connected with appropriate linking words. At times, your points can feel a bit disjointed.',
       'Develop your points further with more detailed explanations and relevant examples to strengthen your argument. This will also help to make your ideas clearer and more comprehensive.',
       'Work on constructing a stronger conclusion that effectively summarizes your main points and gives a clear final opinion. This will make your essay feel more complete.'],
      dtype=object)
gpt_positive_highlights   -> array(['Your es

##### Step 3: Sequence Length Profiling vs. IELTS Standard

In [4]:
df['essay_words'] = df['essay_text'].apply(lambda x: len(str(x).split()))
df['question_words'] = df['question'].apply(lambda x: len(str(x).split()))

print("=== Essay Word Count Statistics ===")
print(df['essay_words'].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]))
print(f"Essays < 150 words: {(df['essay_words'] < 150).sum()}")
print(f"Essays < 220 words: {(df['essay_words'] < 220).sum()}")


=== Essay Word Count Statistics ===
count    8049.000000
mean      294.575972
std        53.950003
min        50.000000
1%        121.000000
5%        222.000000
10%       248.000000
25%       265.000000
50%       291.000000
75%       322.000000
90%       357.000000
99%       444.520000
max       708.000000
Name: essay_words, dtype: float64
Essays < 150 words: 122
Essays < 220 words: 375


##### Step 4: Band Score Distribution & Qualitative Linguistic Audit

In [5]:
print("=== Overall Band Score Frequencies ===")
print(df['overall_band'].value_counts().sort_index())

# Sample syntax across bands
for b in [3.0, 4.5, 6.0, 7.5]:
    subset = df[df['overall_band'] == b]
    if len(subset) > 0:
        row = subset.iloc[0]
        print(f"\n--- Band {b} ({len(row['essay_text'].split())} words) ---")
        print(row['essay_text'][:250].replace('\n', ' '))

=== Overall Band Score Frequencies ===
overall_band
0.0      36
0.5       6
1.0      36
1.5      13
2.0      44
2.5      66
3.0     103
3.5     111
4.0     381
4.5     637
5.0     911
5.5     811
6.0     882
6.5     906
7.0    1021
7.5     956
8.0     686
8.5     389
9.0      54
Name: count, dtype: int64

--- Band 3.0 (197 words) ---
While some individuals believe that quality art can be made by anyone, others think that it demands talent and ability. I personally agree with the latter statement since art not only can be taught but also involves some degree of specific talent.  I

--- Band 4.5 (269 words) ---
Over the last few decades has seen ideas about the western style of clothes, namely,  myriad of people now prioritise to wear up-to-dated clothes rather than traditional clothing.  First and foremost, trigger associated with this trend is that the mo

--- Band 6.0 (271 words) ---
Over the last few decades has seen ideas about the Western style of clothes, namely, the myriad of peo

##### Step 5: Noise Detection (Scraping Artifacts & Corrupted Endings)

In [6]:
import re

has_hashtag = df['essay_text'].str.contains(r'#\w+', regex=True)
has_mention = df['essay_text'].str.contains(r'@\w+', regex=True)
has_telegram = df['essay_text'].str.contains(r't\.me|telegram', case=False, regex=True)
no_terminal_punct = ~df['essay_text'].apply(lambda x: str(x).strip().endswith(('.', '!', '?', '"', "'")))

print(f"Essays with #hashtags (#task2, etc.): {has_hashtag.sum()}")
print(f"Essays with @mentions (@channel):       {has_mention.sum()}")
print(f"Essays with telegram links:              {has_telegram.sum()}")
print(f"Essays cut off mid-sentence:             {no_terminal_punct.sum()}")

Essays with #hashtags (#task2, etc.): 6
Essays with @mentions (@channel):       2
Essays with telegram links:              11
Essays cut off mid-sentence:             406


#### Step 6: Question Frequency & Data Leakage (The Most Critical Step)

In [7]:
print(f"Total essays: {len(df)}")
print(f"Unique questions: {df['question'].nunique()}")

q_counts = df['question'].value_counts()
print("\nTop 3 most repeated questions:")
for q, count in q_counts.head(3).items():
    print(f"  [{count} essays] {q[:90]}...")

Total essays: 8049
Unique questions: 1970

Top 3 most repeated questions:
  [383 essays] Living in a country where you have to speak a foreign language can cause serious social pr...
  [249 essays] Some people believe that professionals, such as doctors and engineers, should be required ...
  [204 essays] Some people believe that it is good to share as much information as possible in scientific...
